# STANCEOSAURUS BENCHMARK: SITPCL-FOT + SPARSEMAX (PROPOSED OPTIMAL METHOD)
Notebook này hiện thực giải pháp nâng tiến **SITPCL-FoT + Sparsemax** (*Speaker Interaction and Target-aware Prototypical Contrastive Learning with Focused Transformer Attention and Sparsemax Activation*) trên tập dữ liệu hội thoại đa bên **Stanceosaurus Multi-party English** (`train_conversations.json`, `dev_conversations.json`, `test_conversations.json`).

### 🌟 SỰ CỘNG HƯỞNG ĐỘC ĐÁO GIỮA FOT VÀ SPARSEMAX:
1. **Hạn chế của Softmax truyền thống:** Luôn gán xác suất $> 0$ cho tất cả các lượt nói trong quá khứ, khiến vector ngữ cảnh bị nhiễu bởi các phát ngôn lạc đề hoặc của người đối thoại khác.
2. **Sparsemax Activation (Martins & Astudillo, ICML 2016):** Chiếu điểm số chú ý lên hình học đơn chuẩn (*Euclidean simplex*), cho phép **cắt thẳng trọng số attention của các lượt thoại không liên quan về CHÍNH XÁC 0.0**.
3. **Focused Transformer (FoT Contrastive Loss):** Sử dụng InfoNCE ở mức vi mô để kéo Query gần với Key của **CÙNG người nói** và đẩy Key của **người đối thoại KHÁC** ra xa.
   $\rightarrow$ Khi điểm số của người khác bị FoT đẩy xuống dưới ngưỡng, **Sparsemax sẽ triệt tiêu hoàn toàn trọng số chú ý của họ về 0**, giúp mô hình đạt độ tập trung tuyệt đối vào lập trường nhất quán của người nói!
$$\mathcal{L}_{total} = \mathcal{L}_{Focal} + \alpha \cdot \mathcal{L}_{TPC} + \beta \cdot \mathcal{L}_{FoT}$$

### ⚖️ ĐỒNG BỘ ĐỐI SOÁT CÔNG BẰNG (FAIR PROTOCOL):
- **Cùng Backbone:** `vinai/bertweet-base` đồng bộ tuyệt đối với Baseline EMNLP 2022 và SITPCL ACL 2025.
- **Cùng Dữ liệu & Split Unseen Claims:** Dùng chung 100% dữ liệu đã cân bằng độ sâu ($K=6$).
- **Cùng Effective Batch Size:** Batch size 16 với tích lũy gradient 2 bước = 32.
- **Bảng đối sánh 3 bên:** Tự động tổng hợp kết quả của **BERTweet Baseline vs SITPCL vs SITPCL-FoT + Sparsemax**.

### 💡 CƠ CHẾ CHẠY TEST NHANH VÀ TREO MÁY QUA ĐÊM:
- **Chạy thử siêu tốc (Smoke Test):** Đặt `DEBUG_MODE = True` ở Cell 2. Code sẽ nạp 40 mẫu/lớp và chạy 1 epoch trong ~30-45 giây.
- **Treo máy qua đêm:** Đổi thành `DEBUG_MODE = False`, sau đó nhấn **Save Version** -> **Save & Run All (Commit)** trên Kaggle.

In [ ]:
# ==============================================================================
# STEP 1: CÀI ĐẶT THƯ VIỆN & TỰ ĐỘNG DÒ TÌM ĐƯỜNG DẪN DỮ LIỆU
# ==============================================================================
!pip install -q transformers openpyxl emoji python-box scikit-learn

import os
import glob
import json
import math
import random
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score, accuracy_score
from tqdm.auto import tqdm
from box import Box

# Tự động dò tìm file dữ liệu train_conversations.json trên Kaggle hoặc Local
found_files = glob.glob('/kaggle/input/**/train_conversations.json', recursive=True) + \
              glob.glob('./**/train_conversations.json', recursive=True) + \
              glob.glob('D:/KLTN/**/train_conversations.json', recursive=True) + \
              glob.glob('d:/Khóa Luận Tốt Nghiệp/**/train_conversations.json', recursive=True)

if not found_files:
    raise FileNotFoundError('❌ Không tìm thấy train_conversations.json! Hãy kiểm tra Input Dataset trên Kaggle.')

DATA_DIR = os.path.dirname(found_files[0])
print(f'--> ✅ Đã kết nối dữ liệu Stanceosaurus Reconstructed tại: {DATA_DIR}')


In [ ]:
# ==============================================================================
# STEP 2: CẤU HÌNH SIÊU THAM SỐ SITPCL-FOT + SPARSEMAX
# ==============================================================================
DEBUG_MODE = False  # <-- ĐỔI THÀNH False KHI MUỐN TREO MÁY QUA ĐÊM TRÊN KAGGLE!

config = Box({
    'data_dir': DATA_DIR,
    'debug_mode': DEBUG_MODE,
    'debug_samples_per_class': 40,        # 40 mẫu/lớp khi chạy thử siêu tốc
    'model_name': 'vinai/bertweet-base',  # Đồng bộ chính xác backbone BERTweet
    'num_classes': 3,                     # 3: Supporting, Refuting, Other
    'max_seq_len': 128,
    'batch_size': 16,                     # Đồng bộ batch size = 16
    'accum_steps': 2,                     # Effective batch size = 32
    'learning_rate': 2e-5,                # Đồng bộ learning rate = 2e-5
    'weight_decay': 0.01,
    'num_epochs': 1 if DEBUG_MODE else 25,  # Trần epoch cao, dừng thật bằng Early Stopping (xem early_stop_patience)
    'early_stop_patience': 4,             # Dừng sớm nếu Dev Macro-F1 không cải thiện sau 4 epoch liên tiếp
    'warmup_ratio': 0.1,
    'gru_hidden': 768,
    'gru_layer': 1,
    
    # === SIÊU THAM SỐ PROTOTYPICAL CONTRASTIVE LOSS (TPCL) ===
    'alpha': 0.5,                         # Trọng số Target Contrastive Loss (TPCL)
    'tau': 0.1,                           # Nhiệt độ TPCL
    
    # === SIÊU THAM SỐ FOCUSED TRANSFORMER (FoT) ===
    'use_fot': True,                      # Bật học tương phản Attention Query/Key của người nói
    'beta': 0.5,                          # Trọng số FoT Loss
    'tau_fot': 0.1,                       # Nhiệt độ InfoNCE trong FoT
    'att_temp': 1.0,                      # Temperature cho Attention tương tác
    
    # === SIÊU THAM SỐ SPARSEMAX ACTIVATION ===
    'use_sparsemax': True,                # True: Dùng Sparsemax (cắt 0 tuyệt đối) | False: Softmax
    
    # === CLASSIFICATION LOSS ===
    'loss_type': 'focal',                 # Đồng bộ Class-Balanced Focal Loss với baseline
    'focal_gamma': 2.0,
    'use_amp': True,                      # Mixed Precision FP16
    'seed': 42,
    'output_dir': './',
    'best_model_path': './best_sitpcl_fot_bertweet_stanceosaurus.pt'
})

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.seed)
config.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

if config.debug_mode:
    print('⚠️ CHẾ ĐỘ: [DEBUG MODE] - Chạy thử nhanh trong 30-45 giây!')
else:
    print('🔥 CHẾ ĐỘ: [FULL TRAINING] - Chạy toàn bộ dữ liệu qua đêm!')
print(f'--> 🚀 Thiết bị: {config.device} | Max Epochs: {config.num_epochs} (early stop patience={config.early_stop_patience}) | Model: SITPCL-FoT ({config.model_name})')
print(f'--> 🎯 FoT: {config.use_fot} (beta={config.beta}, tau_fot={config.tau_fot}) | Sparsemax: {config.use_sparsemax} | TPCL alpha: {config.alpha}')


In [ ]:
# ==============================================================================
# STEP 3: NẠP DỮ LIỆU HỘI THOẠI JSON ĐA BÊN (MULTI-PARTY CONVERSATIONS)
# ==============================================================================
def load_dialogue_json(filename):
    file_path = os.path.join(config.data_dir, filename)
    with open(file_path, 'r', encoding='utf-8') as f:
        samples = json.load(f)
        
    dialogues = []
    for s in samples:
        dialogues.append({
            'target': str(s['target']).strip(),
            'sentences': s['sentences'],
            'speakers': s['speakers'],
            'label': int(s['label'])
        })
    return pd.DataFrame(dialogues)

print('[*] Đang nạp dữ liệu hội thoại JSON đa bên cho SITPCL-FoT...')
train_df = load_dialogue_json('train_conversations.json')
dev_df = load_dialogue_json('dev_conversations.json')
test_df = load_dialogue_json('test_conversations.json')

if config.debug_mode:
    k = config.debug_samples_per_class
    train_df = train_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), k), random_state=config.seed)).reset_index(drop=True)
    dev_df = dev_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max(k//2, 5)), random_state=config.seed)).reset_index(drop=True)
    test_df = test_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max(k//2, 5)), random_state=config.seed)).reset_index(drop=True)
    print(f'⚠️ [DEBUG MODE] Đã trích mẫu: Train={len(train_df)}, Dev={len(dev_df)}, Test={len(test_df)}')
else:
    print(f'--> [FULL DATA] Train : {len(train_df):,} hội thoại | {train_df["target"].nunique()} targets')
    print(f'--> [FULL DATA] Dev   : {len(dev_df):,} hội thoại | {dev_df["target"].nunique()} targets')
    print(f'--> [FULL DATA] Test  : {len(test_df):,} hội thoại | {test_df["target"].nunique()} targets')

label_names = ['Supporting', 'Refuting', 'Other']
dist_df = pd.DataFrame({
    'Train': train_df['label'].value_counts().sort_index(),
    'Dev': dev_df['label'].value_counts().sort_index(),
    'Test': test_df['label'].value_counts().sort_index()
}, index=range(len(label_names)))
dist_df.index = label_names
display(dist_df)


In [ ]:
# ==============================================================================
# STEP 4: PROMPT TOKENIZATION CHO BERTWEET & SITPCL-FOT DATALOADER
# ==============================================================================
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=False)
mask_token = tokenizer.mask_token  # '<mask >' cho RoBERTa/BERTweet
mask_token_id = tokenizer.mask_token_id

print(f'--> Tokenizer mask token: "{mask_token}" (ID: {mask_token_id})')

class SITPCLStanceDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.data = df.to_dict('records')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        sentences = item['sentences']
        speakers = item['speakers']
        target = item['target']
        
        utterances = []
        for s_idx, sen in enumerate(sentences):
            prompt = f'user {speakers[s_idx]} says: {sen} . Stance towards {target} is {mask_token}'
            enc = self.tokenizer(
                prompt,
                max_length=self.max_len,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            input_ids = enc['input_ids'].squeeze(0)
            att_mask = enc['attention_mask'].squeeze(0)
            
            mask_indices = (input_ids == mask_token_id).nonzero(as_tuple=True)[0]
            mask_idx = mask_indices[0].item() if len(mask_indices) > 0 else (att_mask.sum().item() - 2)
            
            utterances.append({
                'input_ids': input_ids,
                'attention_mask': att_mask,
                'mask_idx': mask_idx
            })
            
        return {
            'utterances': utterances,
            'speakers': speakers,
            'target': target,
            'label': torch.tensor(item['label'], dtype=torch.long)
        }

def sitpcl_collate_fn(batch):
    all_input_ids = []
    all_att_masks = []
    all_mask_indices = []
    dia_idx = []
    speakers_list = []
    targets_list = []
    labels_list = []

    curr_idx = 0
    for sample in batch:
        n_utts = len(sample['utterances'])
        dia_idx.append((curr_idx, curr_idx + n_utts))
        curr_idx += n_utts

        for utt in sample['utterances']:
            all_input_ids.append(utt['input_ids'])
            all_att_masks.append(utt['attention_mask'])
            all_mask_indices.append(utt['mask_idx'])

        speakers_list.append(sample['speakers'])
        targets_list.append(sample['target'])
        labels_list.append(sample['label'])

    return {
        'input_ids': torch.stack(all_input_ids),
        'attention_mask': torch.stack(all_att_masks),
        'mask_indices': torch.tensor(all_mask_indices, dtype=torch.long),
        'dia_idx': dia_idx,
        'speakers': speakers_list,
        'targets': targets_list,
        'labels': torch.stack(labels_list)
    }

train_dataset = SITPCLStanceDataset(train_df, tokenizer, config.max_seq_len)
dev_dataset = SITPCLStanceDataset(dev_df, tokenizer, config.max_seq_len)
test_dataset = SITPCLStanceDataset(test_df, tokenizer, config.max_seq_len)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, collate_fn=sitpcl_collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=sitpcl_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=sitpcl_collate_fn)

print('--> ✅ DataLoader SITPCL-FoT đã khởi tạo xong!')


In [ ]:
# ==============================================================================
# STEP 5: KIẾN TRÚC MÔ HÌNH: SPARSEMAX ACTIVATION, FOT ATTENTION & TPCL
# ==============================================================================
class SparsemaxFunction(torch.autograd.Function):
    """Thuật toán Sparsemax chiếu lên Euclidean simplex (Martins & Astudillo, 2016)
    Hỗ trợ vi phân chính xác (differentiable) cho backpropagation và Mixed Precision FP16."""
    @staticmethod
    def forward(ctx, input, dim=-1):
        ctx.dim = dim
        dim_size = input.size(dim)
        
        # 1. Sắp xếp điểm số theo thứ tự giảm dần
        sorted_input, _ = torch.sort(input, descending=True, dim=dim)
        input_cumsum = torch.cumsum(sorted_input, dim=dim)
        
        # 2. Tạo vector chỉ số k (1..K) để broadcast
        shape = [1] * input.dim()
        shape[dim] = dim_size
        k_indices = torch.arange(1, dim_size + 1, device=input.device, dtype=input.dtype).view(*shape)
        
        # 3. Xác định tập hỗ trợ Support: 1 + k * z_(k) > cumsum(z)
        support = (1 + k_indices * sorted_input) > input_cumsum
        k_z = support.sum(dim=dim, keepdim=True).to(dtype=input.dtype)
        
        # 4. Ngưỡng cắt tau(z) = (sum_{j=1}^{k(z)} z_(j) - 1) / k(z)
        tau_z = (torch.gather(input_cumsum, dim, (k_z - 1).long()) - 1) / k_z
        
        # 5. Output sparsemax = max(0, z - tau(z))
        output = torch.clamp(input - tau_z, min=0.0)
        ctx.save_for_backward(output)
        return output

    @staticmethod
    def backward(ctx, grad_output):
        output, = ctx.saved_tensors
        dim = ctx.dim
        
        # Mask các vị trí có xác suất > 0
        nonzeros = (output > 0).to(dtype=grad_output.dtype)
        num_nonzeros = nonzeros.sum(dim=dim, keepdim=True)
        
        # Tính trung bình gradient trên tập Support
        sum_grad = (grad_output * nonzeros).sum(dim=dim, keepdim=True)
        mean_grad = sum_grad / torch.clamp(num_nonzeros, min=1.0)
        
        # Gradient chuẩn: 1_{p > 0} * (grad - mean_grad)
        grad_input = nonzeros * (grad_output - mean_grad)
        return grad_input, None

class Sparsemax(nn.Module):
    def __init__(self, dim=-1):
        super().__init__()
        self.dim = dim

    def forward(self, input):
        return SparsemaxFunction.apply(input, self.dim)

class FoTAttention(nn.Module):
    """Attention Module hỗ trợ cả Softmax và Sparsemax, trích xuất Query và Key cho FoT"""
    def __init__(self, hidden_dim, temperature=1.0, use_sparsemax=True):
        super().__init__()
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.value_proj = nn.Linear(hidden_dim, hidden_dim)
        self.scale = hidden_dim ** -0.5
        self.temperature = temperature
        self.use_sparsemax = use_sparsemax
        
        if use_sparsemax:
            self.act = Sparsemax(dim=-1)
        else:
            self.act = nn.Softmax(dim=-1)

    def forward(self, query, keys, values):
        if query.dim() == 1:
            query = query.unsqueeze(0)
        q = self.query_proj(query)         # (B, D)
        k = self.key_proj(keys)            # (L, D)
        v = self.value_proj(values)        # (L, D)
        
        attn_scores = (torch.matmul(q, k.transpose(-2, -1)) * self.scale) / self.temperature
        attn_weights = self.act(attn_scores) # Áp dụng Sparsemax hoặc Softmax
        output = torch.matmul(attn_weights, v)
        out = output.squeeze(0) if output.size(0) == 1 else output
        return out, q.squeeze(0), k        # Trả về out, q và k để tính FoT Contrastive Loss

class SSE_FoT(nn.Module):
    """Speaker Stance Extractor với Focused Transformer (FoT) & Sparsemax Attention"""
    def __init__(self, hidden_dim=768, temperature=1.0, tau_fot=0.1, use_fot=True, use_sparsemax=True):
        super().__init__()
        self.linear_intra = nn.Linear(hidden_dim * 2, hidden_dim)
        self.linear_inter = nn.Linear(hidden_dim, hidden_dim)
        self.attention_intra = FoTAttention(hidden_dim, temperature=temperature, use_sparsemax=use_sparsemax)
        self.attention_inter = FoTAttention(hidden_dim, temperature=temperature, use_sparsemax=use_sparsemax)
        self.tau_fot = tau_fot
        self.use_fot = use_fot

    def forward(self, utterances, speakers):
        device = utterances.device
        V_lst = []
        last_speaker_idx = dict()
        fot_losses = []

        speakers_tensor = torch.tensor(speakers, device=device)

        for i in range(len(speakers)):
            speaker_id = speakers[i]
            if speaker_id not in last_speaker_idx:
                V_lst.append(utterances[i])
            else:
                prev_idx = last_speaker_idx[speaker_id]
                vh_concat = torch.cat((V_lst[prev_idx], utterances[i]), dim=-1)
                q_intra = self.linear_intra(vh_concat)
                c = utterances[:i+1]
                
                # 1. Intra-speaker Attention (áp dụng Sparsemax lọc sạch người nói lạc đề)
                v_intra, q_vec, k_mat = self.attention_intra(q_intra, c, c)

                # ==========================================================
                # TÍNH TOÁN FOCUSED TRANSFORMER (FoT) CONTRASTIVE LOSS
                # ==========================================================
                if self.training and self.use_fot:
                    # Positive keys: Các lượt thoại trước đó của CÙNG người nói (j < i)
                    hist_speakers = speakers_tensor[:i+1]
                    pos_mask = (hist_speakers == speaker_id)
                    pos_mask[i] = False  # Bỏ qua chính lượt thoại hiện tại
                    
                    # Negative keys: Các lượt thoại của người đối thoại KHÁC
                    neg_mask = (hist_speakers != speaker_id)

                    if pos_mask.any() and neg_mask.any():
                        q_norm = F.normalize(q_vec.unsqueeze(0), p=2, dim=-1) # (1, D)
                        k_norm = F.normalize(k_mat, p=2, dim=-1)              # (i+1, D)
                        
                        sim = torch.matmul(q_norm, k_norm.T).squeeze(0) / self.tau_fot # (i+1,)
                        pos_sim = sim[pos_mask]
                        all_sim = sim[pos_mask | neg_mask]
                        
                        # InfoNCE Loss: Kéo gần cùng speaker, đẩy xa speaker khác
                        step_fot_loss = - (torch.logsumexp(pos_sim, dim=0) - torch.logsumexp(all_sim, dim=0))
                        fot_losses.append(step_fot_loss)

                # 2. Inter-speaker Attention
                q_inter = self.linear_inter(utterances[i])
                k = torch.stack([V_lst[j] for j in range(prev_idx, i)]) if i > prev_idx else utterances[i].unsqueeze(0)
                v_inter, _, _ = self.attention_inter(q_inter, k, k) if len(k) > 0 else (torch.zeros_like(q_inter), None, None)

                V_lst.append(v_intra + v_inter)
                
            last_speaker_idx[speaker_id] = i

        total_fot_loss = torch.stack(fot_losses).mean() if len(fot_losses) > 0 else torch.tensor(0.0, device=device)
        return torch.stack(V_lst), total_fot_loss

def compute_target_prototypes(H_final, targets):
    """Tính toán Prototype vector cho từng target duy nhất trong mini-batch"""
    tar_to_utts = defaultdict(list)
    for i, t in enumerate(targets):
        tar_to_utts[t].append(H_final[i][-1])
        
    unique_targets = list(tar_to_utts.keys())
    prototypes = []
    for t in unique_targets:
        p_t = torch.stack(tar_to_utts[t]).mean(dim=0)
        prototypes.append(p_t)
    prototypes = torch.stack(prototypes)
    return prototypes, unique_targets

def target_prototypical_loss(H_final, targets, tau=0.1):
    """Target-aware Prototypical Contrastive Loss (TPCL chuẩn từ ACL 2025)"""
    if len(H_final) <= 1:
        return torch.tensor(0.0, device=H_final[0].device)
    
    prototypes, unique_targets = compute_target_prototypes(H_final, targets)
    prototypes = F.normalize(prototypes, p=2, dim=-1)   # [K, D]
    embeddings = torch.stack([F.normalize(h[-1], p=2, dim=-1) for h in H_final]) # [B, D]
    
    tar_map = {t: idx for idx, t in enumerate(unique_targets)}
    target_labels = torch.tensor([tar_map[t] for t in targets], device=embeddings.device)
    
    sim_matrix = torch.matmul(embeddings, prototypes.T) / tau  # [B, K]
    loss = F.cross_entropy(sim_matrix, target_labels)
    return loss

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weights=None):
        super().__init__()
        self.gamma = gamma
        self.weights = weights

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.weights, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# Tính class weights đồng bộ với Baseline và SITPCL
counts = [sum(train_df['label'] == c) for c in range(config.num_classes)]
class_weights = torch.tensor([1.0 / (max(c, 1) ** 0.5) for c in counts], dtype=torch.float).to(config.device)
class_weights = class_weights / class_weights.sum()

class SITPCL_FoT_BERTweetModel(nn.Module):
    def __init__(self, config, class_weights):
        super().__init__()
        self.config = config
        self.backbone = AutoModel.from_pretrained(config.model_name)
        hidden_size = self.backbone.config.hidden_size
        self.gru = nn.GRU(input_size=hidden_size, hidden_size=config.gru_hidden, num_layers=config.gru_layer, batch_first=True)
        self.sse = SSE_FoT(
            hidden_dim=config.gru_hidden,
            temperature=config.att_temp,
            tau_fot=config.tau_fot,
            use_fot=config.use_fot,
            use_sparsemax=config.use_sparsemax
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(config.gru_hidden, config.num_classes)
        )
        self.focal_criterion = FocalLoss(gamma=config.focal_gamma, weights=class_weights)

    def forward(self, input_ids, attention_mask, mask_indices, dia_idx, speakers, targets, labels=None):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        
        batch_indices = torch.arange(last_hidden.size(0), device=last_hidden.device)
        mask_reps = last_hidden[batch_indices, mask_indices, :]
        
        H_final = []
        dialogue_representations = []
        batch_fot_losses = []
        
        for d_id, (st, ed) in enumerate(dia_idx):
            d_reps = mask_reps[st:ed]
            o, _ = self.gru(d_reps.unsqueeze(0))
            o = o.squeeze(0)
            v, dia_fot_loss = self.sse(o, speakers[d_id])
            H_final.append(v)
            dialogue_representations.append(v[-1])
            if self.training and dia_fot_loss > 0:
                batch_fot_losses.append(dia_fot_loss)
            
        stance_features = torch.stack(dialogue_representations)
        logits = self.classifier(stance_features)
        
        loss = None
        loss_dict = {}
        if labels is not None:
            focal_loss = self.focal_criterion(logits, labels)
            cl_loss = target_prototypical_loss(H_final, targets, tau=self.config.tau)
            fot_loss = torch.stack(batch_fot_losses).mean() if len(batch_fot_losses) > 0 else torch.tensor(0.0, device=logits.device)
            
            loss = focal_loss + self.config.alpha * cl_loss + self.config.beta * fot_loss
            loss_dict = {
                'total': loss.item(),
                'focal': focal_loss.item(),
                'tpcl': cl_loss.item(),
                'fot': fot_loss.item()
            }
            
        return loss, logits, loss_dict

model = SITPCL_FoT_BERTweetModel(config, class_weights).to(config.device)
print(f'--> ✅ Đã khởi tạo thành công SITPCL-FoT (Sparsemax: {config.use_sparsemax}) trên Backbone BERTweet!')


In [ ]:
# ==============================================================================
# STEP 6: THIẾT LẬP OPTIMIZER & SCHEDULER
# ==============================================================================
optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
total_steps = (len(train_loader) // config.accum_steps) * config.num_epochs
warmup_steps = int(total_steps * config.warmup_ratio)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=max(total_steps, 1))
scaler = torch.cuda.amp.GradScaler(enabled=config.use_amp)

print(f'--> ✅ Tổng số bước huấn luyện SITPCL-FoT: {total_steps} (Warmup: {warmup_steps})')


In [ ]:
# ==============================================================================
# STEP 7: VÒNG LẶP HUẤN LUYỆN VÀ VALIDATION (LOG CHI TIẾT 3 LOSS)
# ==============================================================================
def evaluate_sitpcl_fot(model, loader, target_names):
    model.eval()
    all_preds, all_labels = [], []
    cls_list = list(range(len(target_names)))
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(config.device)
            attention_mask = batch['attention_mask'].to(config.device)
            mask_indices = batch['mask_indices'].to(config.device)
            labels = batch['labels'].to(config.device)

            with torch.cuda.amp.autocast(enabled=config.use_amp):
                _, logits, _ = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    mask_indices=mask_indices,
                    dia_idx=batch['dia_idx'],
                    speakers=batch['speakers'],
                    targets=batch['targets'],
                    labels=None
                )
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, labels=cls_list, average='macro', zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    per_class_f1 = f1_score(all_labels, all_preds, labels=cls_list, average=None, zero_division=0)
    report = classification_report(all_labels, all_preds, labels=cls_list, target_names=target_names, zero_division=0, output_dict=True)
    return macro_f1, acc, per_class_f1, report

best_dev_f1 = 0.0
patience_counter = 0
history = []
best_model_path = config.best_model_path

print('\n[*] BẮT ĐẦU TRAINING SITPCL-FOT + SPARSEMAX (BERTWEET BACKBONE)...')
for epoch in range(1, config.num_epochs + 1):
    model.train()
    total_loss, total_focal, total_tpcl, total_fot = 0.0, 0.0, 0.0, 0.0
    optimizer.zero_grad()

    loop = tqdm(train_loader, desc=f'Epoch {epoch}/{config.num_epochs}')
    for step, batch in enumerate(loop):
        input_ids = batch['input_ids'].to(config.device)
        attention_mask = batch['attention_mask'].to(config.device)
        mask_indices = batch['mask_indices'].to(config.device)
        labels = batch['labels'].to(config.device)

        with torch.cuda.amp.autocast(enabled=config.use_amp):
            loss, _, l_dict = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                mask_indices=mask_indices,
                dia_idx=batch['dia_idx'],
                speakers=batch['speakers'],
                targets=batch['targets'],
                labels=labels
            )
            loss = loss / config.accum_steps

        scaler.scale(loss).backward()
        total_loss += loss.item() * config.accum_steps
        total_focal += l_dict.get('focal', 0.0)
        total_tpcl += l_dict.get('tpcl', 0.0)
        total_fot += l_dict.get('fot', 0.0)

        if (step + 1) % config.accum_steps == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        loop.set_postfix({
            'loss': f"{total_loss / (step + 1):.3f}",
            'focal': f"{total_focal / (step + 1):.3f}",
            'tpcl': f"{total_tpcl / (step + 1):.3f}",
            'fot': f"{total_fot / (step + 1):.3f}"
        })

    avg_train_loss = total_loss / len(train_loader)
    dev_f1, dev_acc, dev_per_f1, _ = evaluate_sitpcl_fot(model, dev_loader, label_names)
    
    print(f'--> [Epoch {epoch}] Train Loss: {avg_train_loss:.4f} (Focal: {total_focal/len(train_loader):.3f}, TPCL: {total_tpcl/len(train_loader):.3f}, FoT: {total_fot/len(train_loader):.3f}) | Dev Macro-F1: {dev_f1*100:.2f}% | Dev Acc: {dev_acc*100:.2f}%')
    history.append({'epoch': epoch, 'train_loss': avg_train_loss, 'dev_macro_f1': dev_f1, 'dev_acc': dev_acc})
    
    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f'    🏆 [Lưu Checkpoint Tốt Nhất] Dev F1 đạt: {best_dev_f1*100:.2f}%')
    else:
        patience_counter += 1
        print(f'    ⏳ [Không cải thiện] {patience_counter}/{config.early_stop_patience} epoch liên tiếp (Best: {best_dev_f1*100:.2f}%)')
        if patience_counter >= config.early_stop_patience:
            print(f'    🛑 [EARLY STOPPING] Dừng tại epoch {epoch} vì Dev Macro-F1 không cải thiện sau {config.early_stop_patience} epoch liên tiếp.')
            break


In [ ]:
# ==============================================================================
# STEP 8: ĐÁNH GIÁ TRÊN TẬP TEST ZERO-SHOT & BẢNG ĐỐI SÁNH 3 MÔ HÌNH
# ==============================================================================
print('\n' + '='*85)
print('[*] ĐÁNH GIÁ SITPCL-FOT + SPARSEMAX TRÊN TẬP TEST ZERO-SHOT VỚI CHECKPOINT TỐT NHẤT')
print('='*85)

best_model_path = getattr(config, 'best_model_path', './best_sitpcl_fot_bertweet_stanceosaurus.pt')

if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path, map_location=config.device))
    print(f'--> [✓] Đã nạp thành công checkpoint tốt nhất từ: {best_model_path}')
else:
    print('--> [!] Không tìm thấy file checkpoint đã lưu, sử dụng trọng số hiện tại của model.')

test_f1, test_acc, test_per_f1, test_report_dict = evaluate_sitpcl_fot(model, test_loader, label_names)

model_label = 'SITPCL-FoT + Sparsemax (Ours)' if config.use_sparsemax else 'SITPCL-FoT (Ours)'
loss_desc = f'Focal + {config.alpha}*TPCL + {config.beta}*FoT' + (' [Sparsemax]' if config.use_sparsemax else '')

res_dict = {
    'Model': model_label,
    'Mode': 'DEBUG' if config.debug_mode else 'FULL',
    'Num_Classes': config.num_classes,
    'Loss_Type': loss_desc,
    'Accuracy': round(test_acc * 100, 2),
    'Macro_F1': round(test_f1 * 100, 2)
}

for idx, name in enumerate(label_names):
    if idx < len(test_per_f1):
        res_dict[f'F1_{name}'] = round(test_per_f1[idx] * 100, 2)

results_df = pd.DataFrame([res_dict])
results_df.to_csv('stanceosaurus_sitpcl_fot_results.csv', index=False)

print('\n--- 📊 BẢNG KẾT QUẢ SITPCL-FOT + SPARSEMAX ---')
display(results_df)

report_df = pd.DataFrame(test_report_dict).T
print('\n--- Báo Cáo Phân Loại Chi Tiết Từng Nhãn ---')
display(report_df.round(3))

# TỔNG HỢP BẢNG ĐỐI SÁNH 3 MÔ HÌNH NẾU CÁC FILE CSV KẾT QUẢ ĐÃ TỒN TẠI
all_results = []
if os.path.exists('stanceosaurus_bertweet_results.csv'):
    all_results.append(pd.read_csv('stanceosaurus_bertweet_results.csv'))
if os.path.exists('stanceosaurus_sitpcl_results.csv'):
    all_results.append(pd.read_csv('stanceosaurus_sitpcl_results.csv'))
all_results.append(results_df)

if len(all_results) > 1:
    print('\n' + '='*85)
    print('🔥 BẢNG ĐỐI SÁNH TỔNG HỢP CÁC PHƯƠNG PHÁP TRÊN STANCEOSAURUS ZERO-SHOT:')
    print('='*85)
    summary_compare_df = pd.concat(all_results, ignore_index=True)
    display(summary_compare_df)
    summary_compare_df.to_csv('stanceosaurus_final_benchmark_comparison.csv', index=False)

if config.debug_mode:
    print('\n🎉 [XÁC NHẬN] CHẾ ĐỘ DEBUG ĐÃ CHẠY THÀNH CÔNG!')
    print('👉 BẠN CHỈ CẦN QUAY LẠI CELL 2, ĐỔI `DEBUG_MODE = False`, RỒI COMMIT TREO QUA ĐÊM TRÊN KAGGLE!')
else:
    print('\n✅ [HOÀN TẤT] ĐÃ CHẠY XONG FULL DỮ LIỆU SITPCL-FOT + SPARSEMAX!')
